Dalam suatu rangkaian listrik tertutup digunakan Hukum Kirchhoff untuk menganalisis arus. Terdapat 4 arus yang belum diketahui yaitu:

i1, i2, i3, i4

Berdasarkan analisis rangkaian, diperoleh sistem persamaan linear sebagai berikut:

10i1 + 2i2 - i3 + i4 = 7

-3i1 + 9i2 + i3 - 2i4 = -8

2i1 - i2 + 7i3 + i4 = 6

i1 + 3i2 - 2i3 + 8i4 = 5

In [1]:
import numpy as np

A = np.array([[10,2,-1,1],
              [-3,9,1,-2],
              [2,-1,7,1],
              [1,3,-2,8]], dtype=float)

b = np.array([7,-8,6,5], dtype=float)

Metode Dekomposisi LU

In [2]:
def lu_decomposition(A):
    n = len(A)
    L = np.zeros((n,n))
    U = np.zeros((n,n))

    for i in range(n):
        L[i][i] = 1

        for j in range(i, n):
            U[i][j] = A[i][j] - sum(L[i][k]*U[k][j] for k in range(i))

        for j in range(i+1, n):
            L[j][i] = (A[j][i] - sum(L[j][k]*U[k][i] for k in range(i))) / U[i][i]

    return L, U

def forward_sub(L, b):
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - sum(L[i][j]*y[j] for j in range(i))
    return y

def backward_sub(U, y):
    n = len(y)
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (y[i] - sum(U[i][j]*x[j] for j in range(i+1, n))) / U[i][i]
    return x

L, U = lu_decomposition(A)
y = forward_sub(L, b)
x_lu = backward_sub(U, y)

print("Solusi LU:", x_lu)

Solusi LU: [ 0.76204769 -0.50041688  0.44939136  0.82974821]


Metode Jacobi

In [4]:
def jacobi(A, b, x0, tol=1e-6, max_iter=100):
    n = len(b)
    x = x0.copy()

    for k in range(max_iter):
        x_new = np.zeros(n)
        for i in range(n):
            s = sum(A[i][j]*x[j] for j in range(n) if j != i)
            x_new[i] = (b[i] - s) / A[i][i]

        if np.linalg.norm(x_new - x) < tol:
            return x_new

        x = x_new

    return x

x0 = np.zeros(4)
x_jacobi = jacobi(A, b, x0)

print("Solusi Jacobi:", x_jacobi)

Solusi Jacobi: [ 0.7620478  -0.50041682  0.44939123  0.82974844]


Metode Gauss-Seidel

In [5]:
def gauss_seidel(A, b, x0, tol=1e-6, max_iter=100):
    n = len(b)
    x = x0.copy()

    for k in range(max_iter):
        x_old = x.copy()

        for i in range(n):
            s1 = sum(A[i][j]*x[j] for j in range(i))
            s2 = sum(A[i][j]*x_old[j] for j in range(i+1, n))
            x[i] = (b[i] - s1 - s2) / A[i][i]

        if np.linalg.norm(x - x_old) < tol:
            return x

    return x

x0 = np.zeros(4)
x_gs = gauss_seidel(A, b, x0)

print("Solusi Gauss-Seidel:", x_gs)

Solusi Gauss-Seidel: [ 0.76204766 -0.5004169   0.44939139  0.82974823]


Perbandingan hasil

In [7]:
print("\nPerbandingan hasil:")
print("Dekomposisi LU :", x_lu)
print("Jacobi         :", x_jacobi)
print("Gauss-Seidel   :", x_gs)


Perbandingan hasil:
Dekomposisi LU : [ 0.76204769 -0.50041688  0.44939136  0.82974821]
Jacobi         : [ 0.7620478  -0.50041682  0.44939123  0.82974844]
Gauss-Seidel   : [ 0.76204766 -0.5004169   0.44939139  0.82974823]
